# NACC Dashboard

In [22]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

try:
    from scipy.stats import t as student_t
    _has_scipy = True
except Exception:
    _has_scipy = False


# =========================
# CONFIG
# =========================
csv_path = r"C:\Users\miked\Desktop2\RIT ISTE Data Mining\project\investigator_nacc70.csv"
out_dir = os.path.expanduser("~/Desktop2/nacc_dashboard")
os.makedirs(out_dir, exist_ok=True)
out_pdf = os.path.join(out_dir, "nacc_dashboard.pdf")

site_focus = "8646"
top_subj_n = 30


# =========================
# LOAD AND FILTER
# =========================
df = pd.read_csv(csv_path, low_memory=False).copy()

needed = [
    "NACCID", "NACCADC", "VISITYR", "FORMVER",
    "NACCUDSD", "MOCATOTS", "CDRSUM", "CDRGLOB", "NACCDIED"
]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError("Missing columns: " + ", ".join(missing))

df["FORMVER"] = pd.to_numeric(df["FORMVER"], errors="coerce")
df = df[df["FORMVER"] >= 3].copy()

df["NACCID"] = df["NACCID"].astype(str)
df["NACCADC"] = df["NACCADC"].astype(str)
df["VISITYR"] = pd.to_numeric(df["VISITYR"], errors="coerce")
df["NACCUDSD"] = pd.to_numeric(df["NACCUDSD"], errors="coerce")
df["MOCATOTS"] = pd.to_numeric(df["MOCATOTS"], errors="coerce")
df["CDRSUM"] = pd.to_numeric(df["CDRSUM"], errors="coerce")
df["CDRGLOB"] = pd.to_numeric(df["CDRGLOB"], errors="coerce")
df["NACCDIED"] = pd.to_numeric(df["NACCDIED"], errors="coerce")

df_site = df[df["NACCADC"] == str(site_focus)].copy()
df_ex_site = df[df["NACCADC"] != str(site_focus)].copy()


# =========================
# LAYOUT HELPERS
# =========================
def page_header(fig, header, subhead, x=0.06, y_title=0.975, y_sub=0.945):
    fig.text(x, y_title, header, fontsize=14, fontweight="bold", va="top")
    fig.text(x, y_sub, subhead, fontsize=10, va="top")


def grid_vertical(fig, top=0.86, left=0.10, right=0.95, bottom=0.10, hspace=0.45):
    return fig.add_gridspec(
        nrows=2, ncols=1,
        left=left, right=right, bottom=bottom, top=top,
        hspace=hspace
    )


def grid_full(fig, top=0.86, left=0.10, right=0.95, bottom=0.12):
    return fig.add_gridspec(
        nrows=1, ncols=1,
        left=left, right=right, bottom=bottom, top=top
    )


# =========================
# CORE SUMMARIES
# =========================
def baseline_year_per_subject(d):
    return d.groupby("NACCID")["VISITYR"].min().reset_index(name="baseline_year")


def new_enrollees_by_year(d):
    base = baseline_year_per_subject(d)
    tmp = d[["NACCID"]].drop_duplicates("NACCID").merge(base, on="NACCID", how="left")
    out = tmp.groupby("baseline_year")["NACCID"].nunique().reset_index(name="new_enrollees")
    out = out.rename(columns={"baseline_year": "year"})
    return out


def new_enrollees_by_site_year(d):
    base = baseline_year_per_subject(d)
    subj_site = d[["NACCID", "NACCADC"]].drop_duplicates("NACCID")
    tmp = subj_site.merge(base, on="NACCID", how="left")
    tmp = tmp.dropna(subset=["baseline_year"])

    out = (
        tmp.groupby(["NACCADC", "baseline_year"])["NACCID"]
           .nunique()
           .reset_index(name="new_enrollees")
           .rename(columns={"baseline_year": "year"})
    )
    return out


def deaths_by_year_proxy(d):
    dd = d[["NACCID", "VISITYR", "NACCDIED"]].dropna(subset=["NACCID", "VISITYR"]).copy()
    if dd.empty:
        return pd.DataFrame(columns=["year", "deaths"])

    last = dd.groupby("NACCID")["VISITYR"].max().reset_index(name="year")
    died = dd.groupby("NACCID")["NACCDIED"].max().reset_index(name="died")
    tmp = last.merge(died, on="NACCID", how="left")
    tmp = tmp[tmp["died"] == 1]

    out = tmp.groupby("year")["NACCID"].nunique().reset_index(name="deaths")
    return out


def current_naccudsd_counts(d):
    dd = d[["NACCID", "VISITYR", "NACCUDSD"]].dropna(subset=["NACCID", "VISITYR", "NACCUDSD"]).copy()
    if dd.empty:
        return pd.DataFrame(columns=["naccudsd", "n"])

    dd = dd.sort_values(["NACCID", "VISITYR"])
    last = dd.groupby("NACCID", as_index=False).last()
    out = last.groupby("NACCUDSD")["NACCID"].nunique().reset_index(name="n")
    out = out.rename(columns={"NACCUDSD": "naccudsd"})
    return out


def yearly_box_data(d, metric_col):
    dd = d[["VISITYR", metric_col]].dropna().copy()
    if dd.empty:
        return [], []

    years = sorted(dd["VISITYR"].dropna().unique().tolist())
    data = []
    for y in years:
        vals = dd.loc[dd["VISITYR"] == y, metric_col].dropna().values
        data.append(vals if len(vals) else np.array([]))
    return years, data


# =========================
# MISSING FOLLOW UP VISITS HEATMAP
# =========================
def missing_followup_visits_by_site_year(d):
    x = d[["NACCID", "NACCADC", "VISITYR"]].dropna().copy()
    if x.empty:
        return pd.DataFrame(columns=["NACCADC", "VISITYR", "missing_visits"])

    bounds = x.groupby("NACCID")["VISITYR"].agg(["min", "max"]).reset_index()
    bounds.columns = ["NACCID", "min_year", "max_year"]
    x = x.merge(bounds, on="NACCID", how="left")

    x = x[x["VISITYR"] > x["min_year"]].copy()

    missing_rows = []
    for sid, g in x.groupby("NACCID"):
        site = g["NACCADC"].iloc[0]
        min_y = int(g["min_year"].iloc[0])
        max_y = int(g["max_year"].iloc[0])

        observed = set(g["VISITYR"].astype(int).tolist())
        expected = set(range(min_y + 1, max_y + 1))
        miss = expected - observed

        for y in miss:
            missing_rows.append((site, y))

    if not missing_rows:
        return pd.DataFrame(columns=["NACCADC", "VISITYR", "missing_visits"])

    miss_df = pd.DataFrame(missing_rows, columns=["NACCADC", "VISITYR"])
    out = miss_df.groupby(["NACCADC", "VISITYR"]).size().reset_index(name="missing_visits")
    return out


# =========================
# GLOBAL METRICS FOR UPDATED PAGES
# =========================
def enroll_abs_yoy_change_by_site_year(d):
    """
    For each site-year:
      delta_abs = abs(new_enrollees(year) - new_enrollees(prev year within site))
    """
    c = new_enrollees_by_site_year(d)
    if c.empty:
        return pd.DataFrame(columns=["NACCADC", "VISITYR", "delta_abs"])

    c = c.rename(columns={"year": "VISITYR"}).copy()
    c = c.sort_values(["NACCADC", "VISITYR"])
    c["delta_abs"] = c.groupby("NACCADC")["new_enrollees"].diff().abs()

    out = c.dropna(subset=["delta_abs"])[["NACCADC", "VISITYR", "delta_abs"]].copy()
    return out


def cognitive_composition_over_time(d):
    x = d[["VISITYR", "NACCUDSD", "NACCID"]].dropna().copy()
    x = x[x["NACCUDSD"].isin([1, 2, 3, 4])].copy()
    if x.empty:
        return pd.DataFrame(columns=["VISITYR", "NACCUDSD", "prop"])

    counts = (
        x.groupby(["VISITYR", "NACCUDSD"])["NACCID"]
         .nunique()
         .reset_index(name="n")
    )
    totals = counts.groupby("VISITYR")["n"].sum().reset_index(name="total")
    counts = counts.merge(totals, on="VISITYR", how="left")
    counts["prop"] = counts["n"] / counts["total"]
    return counts


def normal_over_sum234_by_site_year(d):
    """
    For each site-year:
      ratio = n(status=1) / (n(status in 2,3,4))
    If denom is 0, ratio is NaN (will show white).
    """
    x = d[["NACCADC", "VISITYR", "NACCUDSD", "NACCID"]].dropna().copy()
    x = x[x["NACCUDSD"].isin([1, 2, 3, 4])].copy()
    if x.empty:
        return pd.DataFrame(columns=["NACCADC", "VISITYR", "ratio_normal_over_sum234"])

    counts = (
        x.groupby(["NACCADC", "VISITYR", "NACCUDSD"])["NACCID"]
         .nunique()
         .reset_index(name="n")
    )

    normal = counts[counts["NACCUDSD"] == 1][["NACCADC", "VISITYR", "n"]].rename(columns={"n": "n_normal"})
    impaired = counts[counts["NACCUDSD"].isin([2, 3, 4])].groupby(["NACCADC", "VISITYR"])["n"].sum().reset_index(name="n_234")

    tmp = impaired.merge(normal, on=["NACCADC", "VISITYR"], how="left")
    tmp["n_normal"] = tmp["n_normal"].fillna(0)

    denom = tmp["n_234"].replace(0, np.nan)
    tmp["ratio_normal_over_sum234"] = tmp["n_normal"] / denom

    return tmp[["NACCADC", "VISITYR", "ratio_normal_over_sum234"]]


def max_year_to_year_swing_ratio_by_site(d):
    """
    For each site:
      max abs year-to-year change in ratio_normal_over_sum234
    """
    p = normal_over_sum234_by_site_year(d)
    if p.empty:
        return pd.DataFrame(columns=["NACCADC", "max_swing_ratio"])

    p = p.sort_values(["NACCADC", "VISITYR"])
    p["delta"] = p.groupby("NACCADC")["ratio_normal_over_sum234"].diff().abs()

    out = (
        p.groupby("NACCADC")["delta"]
         .max()
         .reset_index(name="max_swing_ratio")
    )
    return out


def moca_cdr_corr_by_site_year(d, min_n=25):
    """
    For each site-year: correlation(MOCA, CDRSUM) using all rows in that site-year.
    NaN if too few obs or zero variance.
    """
    x = d[["NACCADC", "VISITYR", "MOCATOTS", "CDRSUM"]].dropna().copy()
    if x.empty:
        return pd.DataFrame(columns=["NACCADC", "VISITYR", "moca_cdr_corr", "n_obs"])

    rows = []
    for (site, yr), g in x.groupby(["NACCADC", "VISITYR"], sort=False):
        n = len(g)
        if n < min_n:
            rows.append((site, yr, np.nan, n))
            continue
        if g["MOCATOTS"].nunique() < 2 or g["CDRSUM"].nunique() < 2:
            rows.append((site, yr, np.nan, n))
            continue
        r = g["MOCATOTS"].corr(g["CDRSUM"])
        rows.append((site, yr, r, n))

    return pd.DataFrame(rows, columns=["NACCADC", "VISITYR", "moca_cdr_corr", "n_obs"])


def moca_cdr_corr_by_subject_cumulative_year(d, min_visits=3):
    """
    For each subject and each year they have data:
      corr(subject, year) = corr(MOCA, CDRSUM) using all visits with VISITYR <= year.
    If not enough visits yet, NaN (shows white).
    """
    x = d[["NACCID", "VISITYR", "MOCATOTS", "CDRSUM"]].dropna().copy()
    if x.empty:
        return pd.DataFrame(columns=["NACCID", "VISITYR", "moca_cdr_corr"])

    x = x.sort_values(["NACCID", "VISITYR"])

    out_rows = []
    for sid, g in x.groupby("NACCID", sort=False):
        g = g.sort_values("VISITYR")
        years = sorted(g["VISITYR"].unique().tolist())
        for yr in years:
            h = g[g["VISITYR"] <= yr]
            n = len(h)
            if n < min_visits:
                out_rows.append((sid, yr, np.nan))
                continue
            if h["MOCATOTS"].nunique() < 2 or h["CDRSUM"].nunique() < 2:
                out_rows.append((sid, yr, np.nan))
                continue
            r = h["MOCATOTS"].corr(h["CDRSUM"])
            out_rows.append((sid, yr, r))

    return pd.DataFrame(out_rows, columns=["NACCID", "VISITYR", "moca_cdr_corr"])


# =========================
# STATS HELPERS FOR SITE TABLE
# =========================
def welch_t_pvalue(x, y):
    """
    Welch t-test (two-sided). Uses SciPy if available, else normal approximation.
    Returns (t, df, p).
    """
    x = pd.Series(x).dropna().astype(float)
    y = pd.Series(y).dropna().astype(float)
    if len(x) < 2 or len(y) < 2:
        return np.nan, np.nan, np.nan

    mx, my = float(x.mean()), float(y.mean())
    vx = float(x.var(ddof=1))
    vy = float(y.var(ddof=1))
    nx, ny = len(x), len(y)

    denom = math.sqrt(vx / nx + vy / ny) if (vx > 0 or vy > 0) else np.nan
    if not np.isfinite(denom) or denom == 0:
        return np.nan, np.nan, np.nan

    t = (mx - my) / denom

    num = (vx / nx + vy / ny) ** 2
    den = 0.0
    if vx > 0:
        den += (vx / nx) ** 2 / (nx - 1)
    if vy > 0:
        den += (vy / ny) ** 2 / (ny - 1)
    df_ = num / den if den > 0 else np.nan

    if not np.isfinite(df_):
        return t, df_, np.nan

    if _has_scipy:
        p = 2 * float(student_t.sf(abs(t), df_))
    else:
        p = 2 * (1 - 0.5 * (1 + math.erf(abs(t) / math.sqrt(2))))

    return t, df_, p


def build_site_summary_table(site_focus, df_all):
    """
    Enrollment metric:
      compare per-year abs deltas (site vs all other sites) using Welch t-test.
    Swing metric:
      compare the site's single max_swing_ratio against distribution across sites (z-style).
    """
    delta_all = enroll_abs_yoy_change_by_site_year(df_all)
    delta_site = delta_all[delta_all["NACCADC"].astype(str) == str(site_focus)]["delta_abs"].values
    delta_other = delta_all[delta_all["NACCADC"].astype(str) != str(site_focus)]["delta_abs"].values

    site_avg = float(np.nanmean(delta_site)) if len(delta_site) else np.nan
    glob_avg = float(np.nanmean(delta_other)) if len(delta_other) else np.nan
    t1, df1, p1 = welch_t_pvalue(delta_site, delta_other)

    swing_tbl = max_year_to_year_swing_ratio_by_site(df_all)
    site_row = swing_tbl[swing_tbl["NACCADC"].astype(str) == str(site_focus)]
    site_swing = float(site_row["max_swing_ratio"].iloc[0]) if not site_row.empty else np.nan

    mu = float(swing_tbl["max_swing_ratio"].mean()) if not swing_tbl.empty else np.nan
    sd = float(swing_tbl["max_swing_ratio"].std(ddof=1)) if len(swing_tbl) > 1 else np.nan
    if np.isfinite(site_swing) and np.isfinite(mu) and np.isfinite(sd) and sd > 0:
        z = (site_swing - mu) / sd
        p2 = 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2))))
    else:
        p2 = np.nan

    return pd.DataFrame([
        {
            "metric": "Mean Dif New Subs",
            "mean value": site_avg,
            "global mean": glob_avg,
            "pvalue": p1
        },
        {
            "metric": "Swing Cog Prop (norm/!norm)",
            "mean value": site_swing,
            "global mean": mu,
            "pvalue": p2
        }
    ])


# =========================
# DUMBBELL TABLES (SITE SUBJECTS)
# =========================
def baseline_followup_means_by_subject(d, metric_col):
    dd = d[["NACCID", "VISITYR", metric_col]].dropna().copy()
    if dd.empty:
        return pd.DataFrame(columns=["NACCID", "I_mean", "F_mean", "n_subj"])

    dd = dd.sort_values(["NACCID", "VISITYR"])
    base = dd.groupby("NACCID", as_index=False).first()
    foll = dd.groupby("NACCID", as_index=False).last()

    out = pd.DataFrame({
        "NACCID": base["NACCID"].astype(str),
        "I_mean": base[metric_col].astype(float),
        "F_mean": foll[metric_col].astype(float),
        "n_subj": 1
    })
    return out


df_site_for_dumb = df_site.copy()
subj_visits = df_site_for_dumb.groupby("NACCID").size().sort_values(ascending=False)
top_subjs = subj_visits.head(top_subj_n).index.tolist()
df_site_topsubj = df_site_for_dumb[df_site_for_dumb["NACCID"].isin(top_subjs)].copy()

site_subj_moca = baseline_followup_means_by_subject(df_site_topsubj, "MOCATOTS")
site_subj_cdrsum = baseline_followup_means_by_subject(df_site_topsubj, "CDRSUM")
site_subj_cdrglob = baseline_followup_means_by_subject(df_site_topsubj, "CDRGLOB")


# =========================
# PLOTTING HELPERS
# =========================
def plot_enroll_death_lines(ax, d, title):
    enr = new_enrollees_by_year(d)
    dea = deaths_by_year_proxy(d)

    if d["VISITYR"].dropna().empty:
        ax.text(0.5, 0.5, "No VISITYR data", ha="center", va="center")
        ax.set_title(title)
        ax.axis("off")
        return

    years = np.arange(int(d["VISITYR"].min()), int(d["VISITYR"].max()) + 1)
    tmp = pd.DataFrame({"year": years})
    tmp = tmp.merge(enr, on="year", how="left").merge(dea, on="year", how="left").fillna(0)

    ax.plot(tmp["year"], tmp["new_enrollees"], marker="o", linewidth=1, label="New enrollees")
    ax.plot(tmp["year"], tmp["deaths"], marker="o", linewidth=1, label="Deaths")

    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Count (unique subjects)")

    ax.legend(
        loc="upper right",
        fontsize=8,
        frameon=True,
        borderpad=0.3,
        labelspacing=0.3,
        handlelength=1.2,
        handletextpad=0.5
    )


def plot_naccudsd_bar(ax, d, title):
    mapping = {
        1: "Normal",
        2: "Impaired-not-MCI",
        3: "MCI",
        4: "Dementia"
    }
    cur = current_naccudsd_counts(d)
    cur = cur[cur["naccudsd"].isin([1, 2, 3, 4])].copy()

    if cur.empty:
        ax.text(0.5, 0.5, "No NACCUDSD data", ha="center", va="center")
        ax.set_title(title)
        ax.axis("off")
        return

    cur["label"] = cur["naccudsd"].map(mapping)
    cur = cur.sort_values("naccudsd")

    ax.bar(cur["label"], cur["n"])
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Count (unique subjects)")

    ax.tick_params(axis="x", rotation=30)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")


def plot_year_box(ax, d, metric_col, title, y_label):
    years, data = yearly_box_data(d, metric_col)
    if len(years) == 0:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.set_title(title)
        ax.axis("off")
        return

    ax.boxplot(data, labels=[str(int(y)) for y in years], showfliers=False)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(y_label)

    ax.tick_params(axis="x", rotation=45)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")


def plot_cdrsum_box_with_median(ax, d, title="CDRSUM distribution by year"):
    years, data = yearly_box_data(d, "CDRSUM")
    if len(years) == 0:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.set_title(title)
        ax.axis("off")
        return

    ax.boxplot(data, labels=[str(int(y)) for y in years], showfliers=False)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("CDRSUM")

    med = []
    for arr in data:
        med.append(np.nanmedian(arr) if len(arr) else np.nan)
    ax.plot(np.arange(1, len(years) + 1), med, marker="o", linewidth=1)

    ax.tick_params(axis="x", rotation=45)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")


def plot_missing_followup_heatmap(ax, missing_df, title):
    if missing_df is None or missing_df.empty:
        ax.text(0.5, 0.5, "No missing follow up visits detected", ha="center", va="center")
        ax.set_title(title)
        ax.axis("off")
        return

    pivot = missing_df.pivot(index="NACCADC", columns="VISITYR", values="missing_visits").fillna(0)

    idx_num = pd.to_numeric(pivot.index, errors="coerce")
    if idx_num.notna().any():
        pivot["_idx"] = idx_num
        pivot = pivot.sort_values("_idx").drop(columns=["_idx"])
    else:
        pivot = pivot.sort_index()

    pivot = pivot.sort_index(axis=1)

    im = ax.imshow(pivot.values, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Site")

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels([str(int(x)) for x in pivot.columns], rotation=45)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")

    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index.astype(str).tolist())

    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Missing follow up visits")


def plot_composition_area(ax, d, title):
    comp = cognitive_composition_over_time(d)
    if comp.empty:
        ax.text(0.5, 0.5, "No composition data", ha="center", va="center")
        ax.set_title(title)
        ax.axis("off")
        return

    years = sorted(comp["VISITYR"].unique().tolist())

    labels = [
        "Normal",
        "Impaired-not-MCI",
        "MCI",
        "Dementia"
    ]

    mat = []
    for code in [1, 2, 3, 4]:
        vals = []
        for y in years:
            row = comp[(comp["VISITYR"] == y) & (comp["NACCUDSD"] == code)]
            vals.append(float(row["prop"].iloc[0]) if not row.empty else 0.0)
        mat.append(vals)

    ax.stackplot(years, mat, labels=labels)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Proportion")

    ax.tick_params(axis="x", rotation=45)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")

    ax.legend(loc="upper right", fontsize=8, frameon=True)


def _sorted_index_labels(idx):
    idx = idx.astype(str)
    idx_num = pd.to_numeric(idx, errors="coerce")
    if idx_num.notna().any():
        order = np.argsort(idx_num.fillna(1e18).values)
        return idx.values[order]
    return np.sort(idx.values)


def plot_heatmap_2d(ax, df_long, y_col, x_col, val_col, title, cbar_label):
    """
    2D heatmap with NaNs shown as white.
    """
    if df_long is None or df_long.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title(title)
        ax.axis("off")
        return

    pivot = df_long.pivot(index=y_col, columns=x_col, values=val_col)

    y_labels = _sorted_index_labels(pivot.index.to_series())
    pivot = pivot.reindex(index=y_labels)

    x_num = pd.to_numeric(pivot.columns, errors="coerce")
    if x_num.notna().any():
        cols_sorted = pivot.columns[np.argsort(x_num.values)]
        pivot = pivot.reindex(columns=cols_sorted)
    else:
        pivot = pivot.sort_index(axis=1)

    mat = pivot.to_numpy(dtype=float)
    masked = np.ma.masked_invalid(mat)

    cmap = plt.cm.viridis.copy()
    cmap.set_bad(color="white")

    im = ax.imshow(masked, aspect="auto", cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(y_col)

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels([str(int(c)) if float(c).is_integer() else str(c) for c in pivot.columns], rotation=45)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")

    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index.astype(str).tolist(), fontsize=7)

    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label(cbar_label)


def plot_summary_table(ax, table_df, title):
    ax.axis("off")
    ax.set_title(title, pad=10)

    display_df = table_df.copy()
    for col in ["mean value", "global mean", "pvalue"]:
        if col in display_df.columns:
            display_df[col] = display_df[col].apply(
                lambda x: "" if pd.isna(x) else f"{float(x):.4f}"
            )

    tbl = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        cellLoc="left",
        colLoc="left",
        loc="upper left",
        bbox=[0.00, 0.15, 1.00, 0.80]
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)

    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_text_props(weight="bold")
            cell.set_height(cell.get_height() * 1.15)


def plot_dumbbell(ax, table, group_label, title):
    if table.empty:
        ax.text(0.5, 0.5, "Not enough data", ha="center", va="center")
        ax.set_title(title)
        ax.axis("off")
        return

    t = table.copy()

    t["_group_num"] = pd.to_numeric(t[group_label], errors="coerce")
    if t["_group_num"].notna().any():
        t = t.sort_values(["_group_num", group_label])
    else:
        t = t.sort_values(group_label)
    t = t.drop(columns=["_group_num"])

    y = np.arange(len(t))

    for yi, iv, fv in zip(y, t["I_mean"].values, t["F_mean"].values):
        ax.plot([iv, fv], [yi, yi], linewidth=2, color="darkgray")

    ax.scatter(t["I_mean"].values, y, label="Initial", color="green", marker="o")
    ax.scatter(t["F_mean"].values, y, label="Most recent", color="orange", marker="s")

    ax.set_yticks(y)
    ax.set_yticklabels(t[group_label].astype(str).values)
    ax.set_title(title)
    ax.set_xlabel("Mean score")
    ax.margins(x=0.10)

    ax.legend(
        loc="upper right",
        fontsize=8,
        frameon=True,
        borderpad=0.3,
        labelspacing=0.3,
        handlelength=1.2,
        handletextpad=0.5
    )


# =========================
# BUILD TABLES FOR PAGES
# =========================
missing_df = missing_followup_visits_by_site_year(df)

# GLOBAL page 3
enroll_delta_tbl = enroll_abs_yoy_change_by_site_year(df)

# GLOBAL page 5
case_mix_ratio_tbl = normal_over_sum234_by_site_year(df)
swing_ratio_tbl = max_year_to_year_swing_ratio_by_site(df)  # used for site table

# GLOBAL page 6
corr_site_year_tbl = moca_cdr_corr_by_site_year(df, min_n=25)

# SITE table page
site_summary_tbl = build_site_summary_table(site_focus, df)

# SITE page 11: subject x year cumulative correlation
site_subj_corr_year_tbl_all = moca_cdr_corr_by_subject_cumulative_year(df_site, min_visits=3)
if not site_subj_corr_year_tbl_all.empty:
    subj_counts = df_site.groupby("NACCID").size().sort_values(ascending=False)
    keep_subjs = subj_counts.head(top_subj_n).index.astype(str).tolist()
    site_subj_corr_year_tbl = site_subj_corr_year_tbl_all[site_subj_corr_year_tbl_all["NACCID"].isin(keep_subjs)].copy()
else:
    site_subj_corr_year_tbl = site_subj_corr_year_tbl_all.copy()




Saved PDF: C:\Users\miked/Desktop2/nacc_dashboard\nacc_dashboard.pdf


In [24]:
# =========================
# PDF OUTPUT
# =========================
with PdfPages(out_pdf) as pdf:

    # =========================================================
    # GLOBAL PAGE 1 (keep as is)
    # =========================================================
    fig = plt.figure(figsize=(11, 8.5))
    page_header(
        fig,
        "UDS v3 NACC Dashboard: All Sites",
        "Enrollment and mortality over time | Current cognitive status frequency."
    )
    gs = grid_vertical(fig, hspace=0.45)

    ax = fig.add_subplot(gs[0, 0])
    plot_enroll_death_lines(ax, df, "New enrollees and deaths over time")

    ax = fig.add_subplot(gs[1, 0])
    plot_naccudsd_bar(ax, df, "Current Cognitive Status Frequency")

    pdf.savefig(fig)
    plt.close(fig)

    # =========================================================
    # GLOBAL PAGE 2 (keep as is)
    # =========================================================
    fig = plt.figure(figsize=(11, 8.5))
    page_header(
        fig,
        "UDS v3 NACC Dashboard: All Sites",
        "Missing follow up VISIT counts by site and year."
    )
    gs = grid_full(fig, bottom=0.10)
    ax = fig.add_subplot(gs[0, 0])
    plot_missing_followup_heatmap(ax, missing_df, "Follow up visit completeness (diagnostic)")
    pdf.savefig(fig)
    plt.close(fig)

    # =========================================================
    # GLOBAL PAGE 3 (UPDATED): abs year-to-year change in new enrollees by site and year
    # =========================================================
    sub = (
        "Heatmap: absolute year-to-year change in the number of new enrollees by site\n"
        "X-axis is year, Y-axis is site. Blank cells indicate no prior year for delta."
    )
    fig = plt.figure(figsize=(11, 8.5))
    page_header(fig, "UDS v3 NACC Dashboard: All Sites", sub)
    gs = grid_full(fig, bottom=0.12)
    ax = fig.add_subplot(gs[0, 0])

    plot_heatmap_2d(
        ax,
        enroll_delta_tbl,
        y_col="NACCADC",
        x_col="VISITYR",
        val_col="delta_abs",
        title="Absolute year-to-year change in new enrollees",
        cbar_label="Abs delta (new enrollees)"
    )

    pdf.savefig(fig)
    plt.close(fig)

    # =========================================================
    # GLOBAL PAGE 4 (keep): Cognitive status composition over time (all sites)
    # =========================================================
    fig = plt.figure(figsize=(11, 8.5))
    page_header(
        fig,
        "UDS v3 NACC Dashboard: All Sites",
        "Change in cognitive status composition over time."
    )
    gs = grid_full(fig, bottom=0.12)
    ax = fig.add_subplot(gs[0, 0])
    plot_composition_area(ax, df, "Cognitive status composition over time (all sites)")
    pdf.savefig(fig)
    plt.close(fig)

    # =========================================================
    # GLOBAL PAGE 5 (heatmap): normal/sum(2,3,4) by site and year
    # =========================================================
    sub = (
        "Heatmap: Normal/Not Normal Proportion\n"
        "Blank cells indicate denominator = 0 for that site-year."
    )
    fig = plt.figure(figsize=(11, 8.5))
    page_header(fig, "UDS v3 NACC Dashboard: All Sites", sub)
    gs = grid_full(fig, bottom=0.12)
    ax = fig.add_subplot(gs[0, 0])

    plot_heatmap_2d(
        ax,
        case_mix_ratio_tbl,
        y_col="NACCADC",
        x_col="VISITYR",
        val_col="ratio_normal_over_sum234",
        title="Prop: Normal/Not Normal by site-year",
        cbar_label="Normal/Not Normal"
    )

    pdf.savefig(fig)
    plt.close(fig)

    # =========================================================
    # GLOBAL PAGE 6 (UPDATED): MOCA vs CDRSUM correlation by site-year
    # =========================================================
    sub = (
        "Heatmap: correlation between MOCA and CDRSUM computed within each site-year\n"
        "Blank cells indicate too few observations or no variation for that site-year."
    )
    fig = plt.figure(figsize=(11, 8.5))
    page_header(fig, "UDS v3 NACC Dashboard: All Sites", sub)
    gs = grid_full(fig, bottom=0.12)
    ax = fig.add_subplot(gs[0, 0])

    plot_heatmap_2d(
        ax,
        corr_site_year_tbl,
        y_col="NACCADC",
        x_col="VISITYR",
        val_col="moca_cdr_corr",
        title="MOCA vs CDRSUM correlation by site-year",
        cbar_label="Correlation"
    )

    pdf.savefig(fig)
    plt.close(fig)

    # =========================================================
    # SITE PAGE 1 (keep as is)
    # =========================================================
    fig = plt.figure(figsize=(11, 8.5))
    page_header(
        fig,
        f"UDS v3 NACC Dashboard: Site {site_focus}",
        "Enrollment and mortality over time | Current cognitive status frequency."
    )
    gs = grid_vertical(fig, hspace=0.45)

    ax = fig.add_subplot(gs[0, 0])
    plot_enroll_death_lines(ax, df_site, "New enrollees and deaths over time")

    ax = fig.add_subplot(gs[1, 0])
    plot_naccudsd_bar(ax, df_site, "Current Cognitive Status Frequency")

    pdf.savefig(fig)
    plt.close(fig)

    # =========================================================
    # SITE PAGE 2 (keep as is)
    # =========================================================
    fig = plt.figure(figsize=(11, 8.5))
    page_header(
        fig,
        f"UDS v3 NACC Dashboard: Site {site_focus}",
        "Score distributions by visit year."
    )
    gs = grid_vertical(fig, hspace=0.60)

    ax = fig.add_subplot(gs[0, 0])
    plot_year_box(ax, df_site, "MOCATOTS", "MOCATOTS distribution by visit year (site)", "MOCATOTS")

    ax = fig.add_subplot(gs[1, 0])
    plot_cdrsum_box_with_median(ax, df_site, title="CDRSUM distribution by visit year (site)")

    pdf.savefig(fig)
    plt.close(fig)

    # =========================================================
    # SITE: Cognitive status composition over time (keep as is)
    # =========================================================
    fig = plt.figure(figsize=(11, 8.5))
    page_header(
        fig,
        f"UDS v3 NACC Dashboard: Site {site_focus}",
        "Cognitive status composition over time. Site compared to all sites."
    )
    gs = grid_vertical(fig, hspace=0.55)

    ax = fig.add_subplot(gs[0, 0])
    plot_composition_area(ax, df, "All sites composition over time")

    ax = fig.add_subplot(gs[1, 0])
    plot_composition_area(ax, df_site, f"Site {site_focus} composition over time")

    pdf.savefig(fig)
    plt.close(fig)

    # =========================================================
    # SITE PAGE 11 (UPDATED): subject x year cumulative correlation heatmap
    # =========================================================
    sub = (
        "Heatmap: per-subject correlation between MOCA and CDRSUM computed cumulatively up to each year\n"
        "Blank cells indicate insufficient visits up to that year."
    )
    fig = plt.figure(figsize=(11, 8.5))
    page_header(fig, f"UDS v3 NACC Dashboard: Site {site_focus}", sub)
    gs = grid_full(fig, bottom=0.12)
    ax = fig.add_subplot(gs[0, 0])

    plot_heatmap_2d(
        ax,
        site_subj_corr_year_tbl,
        y_col="NACCID",
        x_col="VISITYR",
        val_col="moca_cdr_corr",
        title="MOCA vs CDRSUM correlation per subject by year (cumulative)",
        cbar_label="Correlation"
    )

    pdf.savefig(fig)
    plt.close(fig)

    # =========================================================
    # DUMBBELLS (leave as is)
    # =========================================================
    fig = plt.figure(figsize=(11, 8.5))
    page_header(
        fig,
        f"UDS v3 NACC Dashboard: Site {site_focus}",
        f"MOCATOTS change per subject (Top {top_subj_n} subjects by visit count)."
    )
    gs = grid_full(fig, left=0.15, bottom=0.12)
    ax = fig.add_subplot(gs[0, 0])
    plot_dumbbell(ax, site_subj_moca, "NACCID", "MOCATOTS: initial vs most recent by subject")
    pdf.savefig(fig)
    plt.close(fig)

    fig = plt.figure(figsize=(11, 8.5))
    page_header(
        fig,
        f"UDS v3 NACC Dashboard: Site {site_focus}",
        f"CDRSUM change per subject (Top {top_subj_n} subjects by visit count)."
    )
    gs = grid_full(fig, left=0.15, bottom=0.12)
    ax = fig.add_subplot(gs[0, 0])
    plot_dumbbell(ax, site_subj_cdrsum, "NACCID", "CDRSUM: initial vs most recent by subject")
    pdf.savefig(fig)
    plt.close(fig)

    fig = plt.figure(figsize=(11, 8.5))
    page_header(
        fig,
        f"UDS v3 NACC Dashboard: Site {site_focus}",
        f"CDRGLOB change per subject (Top {top_subj_n} subjects by visit count)."
    )
    gs = grid_full(fig, left=0.15, bottom=0.12)
    ax = fig.add_subplot(gs[0, 0])
    plot_dumbbell(ax, site_subj_cdrglob, "NACCID", "CDRGLOB: initial vs most recent by subject")
    pdf.savefig(fig)
    plt.close(fig)

print("Saved PDF:", out_pdf)


Saved PDF: C:\Users\miked/Desktop2/nacc_dashboard\nacc_dashboard.pdf
